# Exercise 2: Modify History Frequency


<div class="alert alert-info">

## Exercise: Create a daily history file

Create a new case named `i2000_bgc_hist_daily` using the compset `I2000Clm60BgcCrujra` and resolution `f19_f19_mt233`.

By default CLM writes one monthly history file. Add a second history file that writes daily averages of ``TSA`` (2-m air temperature) and ``TLAI`` (leaf area index).

</div>



<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>
    
**How do I control the output?**

Use CLM namelist variables:

```fortran
hist_fincl1 and hist_fincl2
hist_nhtfrq
```

Remember that if `hist_nhtfrq = 0`, the file contains monthly averages, if `hist_nhtfrq > 0`, the output frequency is given as a number of model time steps, and if `hist_nhtfrq < 0`, the output frequency is given as a number of hours.

</details>
</div>




<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
                
**Create a new case**

Create a new case `i2000_bgc_hist_daily` with the command:

```bash
cd /glade/u/home/$USER/code/my_clm_code/cime/scripts/

./create_newcase --case ~/cases/i2000_bgc_hist_daily \
  --compset I2000Clm60BgcCrujra \
  --res f19_f19_mt233 \
  --run-unsupported
```

**Setup**
    
Invoke `case.setup` with the command:

```bash
cd ~/cases/i2000_bgc_hist_daily
./case.setup
```

**Customize namelists**
    
Edit the file `user_nl_clm` and add the lines:

```fortran
hist_fincl2 = 'TSA','TLAI'
hist_nhtfrq = 0,-24
```
 
**Set run length**
    
Change the run length:

```bash
./xmlchange STOP_N=1
./xmlchange STOP_OPTION=nmonths
```

**Change the wallclock time, job queue, and account number**

Change the wallclock time. If needed, change `job queue` and `account number`. <br>
For instance, to run in the queue `tutorial` and the project number ``UESM0016`` (You should use the project number given for this tutorial), use the command:

```
./xmlchange JOB_WALLCLOCK_TIME=01:00:00
./xmlchange JOB_QUEUE=tutorial
./xmlchange PROJECT=UESM0016 --force
```

**Build and submit**
    
Build the model and submit your job:
```
qcmd -- ./case.build
./case.submit
```

____
    
**Look at your solution**

When the run is complete, go to the archive directory:

```bash
cd /glade/derecho/scratch/$USER/archive/b1850_high_freq/atm/hist
ls
```

In CESM3, CAM history files are separated into averaged and instantaneous files using the suffixes `a` and `i`.

You should see files like:

```text
- h0 
b1850_high_freq.cam.h0a.0001-01.nc
b1850_high_freq.cam.h0i.0001-02-01-00000.nc

- h1
b1850_high_freq.cam.h1i.0001-01-02-00000.nc

- h2
b1850_high_freq.cam.h2a.0001-01-01-10800.nc
...
b1850_high_freq.cam.h2a.0001-01-31-10800.nc

b1850_high_freq.cam.h2i.0001-01-01-10800.nc
...
b1850_high_freq.cam.h2i.0001-01-31-10800.nc
```

The important files for this exercise are:

* `h1i`: daily instantaneous output requested with `fincl2 = 'T:I','Q:I','U:I','V:I'`
* `h2a`: 3-hourly averaged output requested with `fincl3 = 'T:A','Q:A','U:A','V:A'`

You will also see companion instantaneous or averaged files, such as `h0i` or `h2i`. This is expected in CESM3 because CAM history streams are split into `a` and `i` files.

For this exercise, focus on checking:

```bash
ncdump -h b1850_high_freq.cam.h1i.0001-01-02-00000.nc
ncdump -h b1850_high_freq.cam.h2a.0001-01-01-10800.nc
```

The `h1i` file should contain the instantaneous fields. These variables should have:

```text
cell_methods = "time: point" ;
```

The `h2a` files should contain the time-averaged fields. These variables should include:

```text
cell_methods = "time: mean"
```

Check the number of time samples:

```bash
ncdump -h b1850_high_freq.cam.h1i.0001-01-02-00000.nc | grep time
ncdump -h b1850_high_freq.cam.h2a.0001-01-01-10800.nc | grep time
```

The `h1i` file should contain 31 time samples.

Each `h2a` file should contain 8 time samples.

You can also check file sizes with:

```bash
du -skh /glade/derecho/scratch/$USER/archive/b1850_high_freq/atm/hist/*
```
____
    
**# Why do some filenames end with `00000` while others end with `10800`?**

The last field in a CAM history filename is the number of **seconds since midnight** associated with the output time.

For example:

* `00000` corresponds to **00:00:00 UTC (midnight)**.
* `10800` corresponds to **03:00:00 UTC** (3 × 3600 seconds = 3 hours).

For this exercise:

* The `h1i` stream contains one instantaneous sample per day. The sample is written at the beginning of the day, so the filename ends with `00000`.

* The `h2a` stream contains 3-hourly **time-averaged** output. The first average represents the interval from **00:00 to 03:00**, so the timestamp corresponds to the **end of the averaging period**, giving `10800` (03:00 UTC).

Similarly, the remaining time samples in each `h2a` file correspond to averages ending at 06:00, 09:00, ..., up to 24:00.

Notice that there is also an `h2i` file. It contains the corresponding instantaneous output at the same sampling frequency, with timestamps matching the same output times.

</details>
</div>

